# Design a Basic Transformer architecture in order to translate english to french

In [1]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import re

## Tokenizer

In [2]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("t5-small")

## Add the BOS token

tok.add_special_tokens({"bos_token": "<s>"})

/home/mgiovannetti/apprenticeship/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1

In [3]:

print(len(tok), tok.pad_token_id, tok.eos_token_id, tok.bos_token_id)

32101 0 1 32100


In [4]:
EOS_TOKEN=tok.eos_token_id
BOS_TOKEN=tok.bos_token_id
PAD_TOKEN=tok.pad_token_id

## Transformer CONFIG

In [5]:
cfg = { 
    "d_model": 256,
    "n_head": 4,
    "qkv_bias": False,
    "context_size": 10000,
    "vocab_size": len(tok),
    "dropout": 0.2,
    "n_layers": 3,
    "temperature": 1,
    "pad_token": PAD_TOKEN
}

MAX_LEN = 128                                  # same threshold as the length filter

cfgV2 = {
    "d_model": 256,
    "n_head": 4,                               # d_head = 256 / 4 = 64 (even ✅ for RoPE)
    "n_layers": 3,                             # 3 encoder + 3 decoder blocks
    "d_ff": 704,                               # SwiGLU: ≈ 8/3 × 256 = 682 → rounded to a multiple of 64
    "qkv_bias": False,
    "dropout": 0.1,                            # 0.1–0.2 on Tatoeba; raise it if val goes back up early
    "context_size": MAX_LEN + 2,               # max length + <bos>/<eos> margin; sizes the RoPE tables
    "vocab_size": len(tok),
    "pad_token": PAD_TOKEN,
}

## Set the DataSet

In [6]:
import random
pairs = []
with open("fra.txt", encoding="utf-8") as f:
    for line in f:
        en, fr = line.rstrip("\n").split("\t")[:2]       # 3rd column = attribution, ignore it
        pairs.append((en, fr))
random.seed(42); random.shuffle(pairs)

In [7]:
print(len(pairs))

240521


In [8]:
print(pairs[43])

("Don't worry, I won't turn you in to the police.", 'T’en fais pas, je te balancerai pas aux flics.')


In [9]:
train_size = int(0.85 * len(pairs))
test_size = int(0.10 * len(pairs))
val_size = len(pairs) - test_size - train_size

In [10]:
train_data = pairs[:train_size]
test_data = pairs[train_size: train_size + test_size]
val_data = pairs[train_size + test_size:]

In [11]:
print(train_data[0:4])

[('Can he see me?', 'Est-il en mesure de me voir ?'), ("I didn't know you had company.", "J'ignorais que tu avais de la compagnie."), ('Tom walks quickly.', 'Tom marche rapidement.'), ('I suspect both Tom and Mary have houses in Boston.', 'Je soupçonne que Tom et Marie ont tous les deux des maisons à Boston.')]


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [13]:
class EnglishToFrenchDataset(Dataset):
    def __init__(self, data, tokenizer, bos_token, eos_token):
        super().__init__()
        self.data = data
        self.eos_token = eos_token
        self.bos_token = bos_token
        self.english_ids = []
        self.french_ids = []
        for row_english, row_french in data:
            self.english_ids.append(tokenizer.encode(row_english))
            self.french_ids.append(tokenizer.encode(row_french, add_special_tokens=False))


    def __len__(self):
        return len(self.english_ids)

    def __getitem__(self, idx):
        src = torch.tensor(self.english_ids[idx])
        tgt = torch.tensor([self.bos_token] + self.french_ids[idx] + [self.eos_token])
        return src, tgt[:-1], tgt[1:]

In [14]:
train_ds = EnglishToFrenchDataset(data=train_data, tokenizer=tok, bos_token=BOS_TOKEN, eos_token=EOS_TOKEN)
test_ds = EnglishToFrenchDataset(data=test_data, tokenizer=tok, bos_token=BOS_TOKEN, eos_token=EOS_TOKEN)
val_ds = EnglishToFrenchDataset(data=val_data, tokenizer=tok, bos_token=BOS_TOKEN, eos_token=EOS_TOKEN)

In [15]:
def collate_fn(batch, pad_id):
    src, dec_in, dec_out = zip(*batch)
    src = pad_sequence(src, batch_first=True, padding_value=pad_id)
    dec_in = pad_sequence(dec_in, batch_first=True, padding_value=pad_id)
    dec_out = pad_sequence(dec_out, batch_first=True, padding_value=-100)

    return src, dec_in, dec_out

In [16]:
from functools import partial
collate_fn_translator = partial(collate_fn, pad_id = PAD_TOKEN)


In [17]:
batch_size = 64

In [18]:
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn_translator)
test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_translator)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_translator)

In [19]:
print(len(train_dl.dataset))

204442


# Init the Model

In [20]:
from transformer import Transformer, TransformerV2

model = TransformerV2(cfgV2)
model = model.to(device)

In [21]:
loss_fn = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay = 0.01)

n_epochs = 30

In [22]:
from transformers import get_cosine_schedule_with_warmup
total_steps = n_epochs * len(train_dl)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=int(0.05 * total_steps), num_training_steps=total_steps)

In [23]:
@torch.no_grad()
def evaluate(val_dl, model, loss_fn, device="cpu"):
    model.eval()
    loss_score, n = 0.0, 0
    for src, dec_in, dec_out in val_dl:
        src, dec_in, dec_out = src.to(device), dec_in.to(device), dec_out.to(device)
        logits = model(src, dec_in, make_src_mask(src), make_target_mask(dec_in))
        loss = loss_fn(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
        loss_score += loss.item() * src.size(0)
        n += src.size(0)
    model.train()
    return loss_score / n


In [24]:
def make_src_mask(x):
    return (x != PAD_TOKEN)[:, None, None, :]


def make_target_mask(x):
    seq_size = x.size(1)
    pad = (x != PAD_TOKEN)[:, None, None, :]
    causal = torch.tril(torch.ones(seq_size, seq_size, dtype=torch.bool, device=x.device))
    return pad & causal

In [25]:
import matplotlib.pyplot as plt

In [26]:
def training_loop(model, train_dl, loss_fn, optimizer, scheduler, device="cpu"):
    model.train()
    loss_score, n = 0.0, 0
    lrs = []
    for src, dec_in, dec_out in train_dl:
        src, dec_in, dec_out = src.to(device), dec_in.to(device), dec_out.to(device)
        optimizer.zero_grad()
        logits = model(src, dec_in, make_src_mask(src), make_target_mask(dec_in))
        loss = loss_fn(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        loss_score += loss.item() * src.size(0)
        n += src.size(0)
        lrs.append(scheduler.get_last_lr()[0])
    return loss_score / n, lrs




def plot_history(history):
    epochs = range(len(history["train"]))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Loss per epoch (one axis, two series, legend)
    ax1.plot(epochs, history["train"], color="#2a6fdb", lw=2, marker="o", ms=4, label="train")
    ax1.plot(epochs, history["val"], color="#e07b39", lw=2, marker="o", ms=4, label="val")
    best = min(range(len(history["val"])), key=history["val"].__getitem__)
    ax1.axvline(best, color="gray", lw=1, ls="--")
    ax1.annotate(f"best val {history['val'][best]:.3f}", (best, history["val"][best]),
                 xytext=(5, 10), textcoords="offset points", fontsize=9, color="dimgray")
    ax1.set(title="Loss", xlabel="epoch", ylabel="cross-entropy")
    ax1.legend(frameon=False)

    # Learning rate per step (separate chart, different scale)
    ax2.plot(history["lr"], color="#2a6fdb", lw=2)
    ax2.set(title="Learning rate", xlabel="step", ylabel="lr")

    for ax in (ax1, ax2):
        ax.grid(alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()


def train(model, train_dl, val_dl, loss_fn, optimizer, scheduler, n_epochs, device="cpu",
          plot_every=1):
    history = {"train": [], "val": [], "lr": []}
    best_val = float("inf")
    for epoch in range(n_epochs):
        train_loss, lrs = training_loop(model, train_dl, loss_fn, optimizer, scheduler, device)
        val_loss = evaluate(val_dl, model, loss_fn, device)

        history["train"].append(train_loss)
        history["val"].append(val_loss)
        history["lr"].extend(lrs)
        print(f"epoch {epoch:3d} | train {train_loss:.3f} | val {val_loss:.3f} | lr {lrs[-1]:.2e}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), "best_model.pt")

    plot_history(history)
    return history

In [27]:
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} parameters")  

13,827,840 parameters


In [28]:
history = train(model, train_dl, val_dl, loss_fn, optimizer, scheduler, n_epochs=n_epochs, device=device)

epoch   0 | train 4.392 | val 2.694 | lr 3.33e-04
epoch   1 | train 2.459 | val 2.261 | lr 5.00e-04
epoch   2 | train 2.156 | val 2.114 | lr 4.97e-04
epoch   3 | train 2.023 | val 2.049 | lr 4.91e-04
epoch   4 | train 1.943 | val 2.007 | lr 4.82e-04
epoch   5 | train 1.885 | val 1.980 | lr 4.70e-04
epoch   6 | train 1.839 | val 1.960 | lr 4.55e-04
epoch   7 | train 1.801 | val 1.946 | lr 4.39e-04
epoch   8 | train 1.769 | val 1.937 | lr 4.19e-04
epoch   9 | train 1.741 | val 1.930 | lr 3.98e-04
epoch  10 | train 1.716 | val 1.923 | lr 3.75e-04
epoch  11 | train 1.693 | val 1.921 | lr 3.50e-04
epoch  12 | train 1.671 | val 1.913 | lr 3.25e-04
epoch  13 | train 1.652 | val 1.912 | lr 2.98e-04
epoch  14 | train 1.634 | val 1.911 | lr 2.71e-04
epoch  15 | train 1.617 | val 1.910 | lr 2.43e-04
epoch  16 | train 1.601 | val 1.910 | lr 2.16e-04
epoch  17 | train 1.587 | val 1.911 | lr 1.89e-04
epoch  18 | train 1.574 | val 1.912 | lr 1.62e-04
epoch  19 | train 1.562 | val 1.912 | lr 1.37e-04


KeyboardInterrupt: 

In [47]:
test_text = "First things first"

In [30]:
@torch.no_grad()
def translate_text(model, text, max_len=200):
    model.eval()

    src = torch.tensor([tok.encode(text)], dtype=torch.long, device=device)  
    src_mask = make_src_mask(src)

    d_in = torch.tensor([[BOS_TOKEN]], dtype=torch.long, device=device)      

    while d_in.size(1) < max_len:
        tgt_mask = make_target_mask(d_in)
        logits = model(src, d_in, src_mask, tgt_mask)                        
        next_id = logits[:, -1].argmax(dim=-1, keepdim=True)                  
        d_in = torch.cat([d_in, next_id], dim=1)                              
        if next_id.item() == EOS_TOKEN:
            break

    return tok.decode(d_in[0], skip_special_tokens=True)
        




In [48]:
print(translate_text(model, test_text))

Commençons par le commencement!


In [42]:
for i in range(5):
    en, fr = test_data[i]
    print("EN :", en)
    print("REF:", fr)
    print("OUT:", translate_text(model, en), "\n")

EN : That's a load of rubbish.
REF: C'est un tas de bêtises.
OUT: C'est une dente. 

EN : I'm too tired to argue.
REF: Je suis trop fatigué pour me disputer.
OUT: Je suis trop fatigué pour me disputer. 

EN : It's already time to go to sleep.
REF: C'est déjà l'heure de dormir.
OUT: Il est déjà l'heure de dormir. 

EN : Let's get up.
REF: Levons-nous.
OUT: Lève-nous. 

EN : I'm missing a sock.
REF: Il me manque une chaussette.
OUT: Il manque une chaussette. 



## Save a Model

In [53]:
import os
os.makedirs("checkpoints", exist_ok=True)

def save_checkpoint(path, model, cfg, optimizer=None, scheduler=None, epoch=None, val_loss=None):
    torch.save({
        "model": model.state_dict(),
        "cfg": cfgV2,
        "optimizer": optimizer.state_dict() if optimizer else None,
        "scheduler": scheduler.state_dict() if scheduler else None,
        "epoch": epoch,
        "val_loss": val_loss,
    }, path)

# at the end of training
save_checkpoint("checkpoints/transformer_v2_final.pt", model, cfg, optimizer, scheduler,
                epoch=24, val_loss=1.919)

# also save the tokenizer (you added a <bos> token, so its vocabulary differs from t5-small)
tok.save_pretrained("checkpoints/tokenizer")

('checkpoints/tokenizer/tokenizer_config.json',
 'checkpoints/tokenizer/tokenizer.json')

## Load a Model

In [54]:
from transformers import AutoTokenizer
from transformer import TransformerV2
 
ckpt = torch.load("checkpoints/transformer_v2_final.pt", map_location=device, weights_only=False)
tok = AutoTokenizer.from_pretrained("checkpoints/tokenizer")

model = TransformerV2(ckpt["cfg"]).to(device)
model.load_state_dict(ckpt["model"])
model.eval()

print(translate_text(model, "I love cats."))

J'adore les chats.


In [ ]:
batch = next(iter(train_dl))
print(batch)
enc, dec_in, dec_out = batch[0]

tok.decode(dec_in)

NameError: name 'train_dl' is not defined

## Improvement

- Scheduler
- warmup
- gradient clipping